# Moosic — 08. Cross-Algorithm Agreement

**Question:** does the same handful of real musical categories independently re-emerge
across K-Means, Agglomerative, and DBSCAN — or was that just visual pattern-matching on
feature averages? This notebook makes it quantitative: for every pair of algorithms, how
much do their cluster assignments actually agree, using standard metrics (not eyeballing),
plus readable heatmaps.

**On DBSCAN's noise:** points DBSCAN labels `-1` are excluded from agreement scoring against
that pair — noise means "DBSCAN made no cluster claim," so it can't meaningfully agree or
disagree with another method on those points.


## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from scipy.optimize import linear_sum_assignment
from sklearn import set_config
import os

set_config(transform_output="pandas")
os.makedirs("../outputs", exist_ok=True)
RANDOM_STATE = 42

## 2. Load Data & Scale

In [ ]:
df = pd.read_csv("../data/5000_songs.csv")
df.columns = df.columns.str.strip()

features = ['danceability', 'energy', 'acousticness', 'tempo', 'valence',
            'speechiness', 'instrumentalness']

scaler = MinMaxScaler().set_output(transform="pandas")
scaled_all = scaler.fit_transform(df[features])

print(f"Shape: {df.shape}")

## 3. Run All Three Canonical Clusterings

In [ ]:
kmeans = KMeans(n_clusters=8, random_state=RANDOM_STATE, n_init=10)
df['kmeans_cluster'] = kmeans.fit_predict(scaled_all)

agg = AgglomerativeClustering(n_clusters=8, metric='euclidean', linkage='ward')
df['agg_cluster'] = agg.fit_predict(scaled_all)

dbscan = DBSCAN(eps=0.20, min_samples=7)
df['dbscan_cluster'] = dbscan.fit_predict(scaled_all)

print("K-Means:", df['kmeans_cluster'].value_counts().sort_index().tolist())
print("Agglomerative:", df['agg_cluster'].value_counts().sort_index().tolist())
print("DBSCAN:", df['dbscan_cluster'].value_counts().sort_index().tolist())

## 4. Quantify Agreement — ARI & NMI

**Adjusted Rand Index (ARI):** measures label agreement, corrected for chance. 1.0 = identical
partitions, 0.0 = no better than random, negative = worse than random. Doesn't require the
same number of clusters or matching label IDs — it's built for exactly this comparison.

**Normalized Mutual Information (NMI):** measures shared information between two labelings,
0 to 1. Answers a slightly different question than ARI — less sensitive to cluster size
imbalance — worth reporting both rather than picking one.


In [ ]:
def agreement(col_a, col_b, exclude_noise_from=None):
    data = df.copy()
    if exclude_noise_from:
        data = data[data[exclude_noise_from] != -1]
    ari = adjusted_rand_score(data[col_a], data[col_b])
    nmi = normalized_mutual_info_score(data[col_a], data[col_b])
    return ari, nmi

pairs = [
    ("K-Means vs Agglomerative", 'kmeans_cluster', 'agg_cluster', None),
    ("K-Means vs DBSCAN", 'kmeans_cluster', 'dbscan_cluster', 'dbscan_cluster'),
    ("Agglomerative vs DBSCAN", 'agg_cluster', 'dbscan_cluster', 'dbscan_cluster'),
]

results = []
for label, a, b, exclude in pairs:
    ari, nmi = agreement(a, b, exclude)
    results.append({"Comparison": label, "ARI": round(ari, 3), "NMI": round(nmi, 3)})

agreement_df = pd.DataFrame(results)
agreement_df

**How to read this:** ARI around 0.3-0.5+ between two genuinely different algorithms, run
with completely different assumptions, is a real, meaningful signal of shared structure —
not proof of identical results (that would be suspicious, not reassuring), but evidence
they're finding the same underlying shape in the data rather than unrelated groupings.


## 5. Readable Heatmaps — Matched and Reordered

A raw crosstab is unreadable — cluster IDs are arbitrary, so "K-Means cluster 3" has no
inherent relationship to "Agglomerative cluster 3." This reorders each heatmap's rows/columns
using optimal matching (Hungarian algorithm) so that the pair of clusters with the most
shared songs lines up on the diagonal — turning "do these agree" into something you can
actually see, not just read as numbers.


In [ ]:
def matched_heatmap(col_a, label_a, col_b, label_b, exclude_noise_from=None, ax=None):
    data = df.copy()
    if exclude_noise_from:
        data = data[data[exclude_noise_from] != -1]

    ct = pd.crosstab(data[col_a], data[col_b])

    # Hungarian algorithm on negative overlap = maximum-overlap matching
    cost = -ct.values
    row_ind, col_ind = linear_sum_assignment(cost) if ct.shape[0] <= ct.shape[1] else \
                        linear_sum_assignment(cost.T)

    if ct.shape[0] <= ct.shape[1]:
        row_order = row_ind
        col_order = list(col_ind) + [c for c in range(ct.shape[1]) if c not in col_ind]
    else:
        col_order = row_ind  # swapped due to transpose above
        row_order = list(col_ind) + [r for r in range(ct.shape[0]) if r not in col_ind]

    ct_reordered = ct.iloc[row_order, col_order] if ct.shape[0] <= ct.shape[1] else ct.iloc[row_order, :].iloc[:, col_order]

    sns.heatmap(ct_reordered, annot=True, fmt='d', cmap='YlOrBr', ax=ax, cbar=False)
    ax.set_xlabel(label_b)
    ax.set_ylabel(label_a)
    return ct_reordered

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

matched_heatmap('kmeans_cluster', 'K-Means', 'agg_cluster', 'Agglomerative', ax=axes[0])
axes[0].set_title('K-Means vs Agglomerative')

matched_heatmap('kmeans_cluster', 'K-Means', 'dbscan_cluster', 'DBSCAN', exclude_noise_from='dbscan_cluster', ax=axes[1])
axes[1].set_title('K-Means vs DBSCAN (noise excluded)')

matched_heatmap('agg_cluster', 'Agglomerative', 'dbscan_cluster', 'DBSCAN', exclude_noise_from='dbscan_cluster', ax=axes[2])
axes[2].set_title('Agglomerative vs DBSCAN (noise excluded)')

plt.tight_layout()
plt.savefig("../outputs/08_cross_algorithm_heatmaps.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Verdict

*(Fill in after running: do the heatmaps show a clear diagonal — most songs in one method's
cluster landing predominantly in a single corresponding cluster of the other method — or a
scattered pattern? Check the ARI/NMI table above alongside the visual read. Either result is
worth stating plainly for the presentation: a strong diagonal is the strongest evidence in
the whole project that these are real categories, not algorithm artifacts.)*
